In [2]:
import pandas as pd
import os
RAW_DIR = "../04_synthetic_data/raw"
CLEAN_DIR = "../04_synthetic_data/clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

suppliers = pd.read_csv(f"{RAW_DIR}/Suppliers.csv")
products  = pd.read_csv(f"{RAW_DIR}/Product_Master.csv")
sales     = pd.read_csv(f"{RAW_DIR}/Sales.csv", parse_dates=["Sale_Date"])
stock     = pd.read_csv(f"{RAW_DIR}/Stock_Daily.csv", parse_dates=["Date"])
orders    = pd.read_csv(f"{RAW_DIR}/Orders.csv", parse_dates=["Order_Date","Expected_Date","Received_Date"])
requests  = pd.read_csv(f"{RAW_DIR}/Customer_Requests.csv", parse_dates=["Request_Date","Fulfilled_Date"])


issues = []

def check(name, cond_df):
    n = len(cond_df)
    status = "PASS" if n == 0 else f"FLAG ({n} rows)"
    issues.append((name, status))

check("Duplicate Sale_ID", sales[sales.duplicated("Sale_ID")])
check("Sales.Product_ID not in Product_Master", sales[~sales.Product_ID.isin(products.Product_ID)])
check("Sales with non-positive Quantity", sales[sales.Quantity <= 0])
check("Sales.Sale_Date outside data window", sales[(sales.Sale_Date < "2025-01-01") | (sales.Sale_Date > "2025-09-30")])
check("Sales for inactive products", sales.merge(products, on="Product_ID")[lambda x: x.Active == 0])

stock_check = stock.copy()
stock_check["expected_closing"] = stock_check.Opening_Stock + stock_check.Received - stock_check.Sold + stock_check.Adjustments
check("Stock_Daily balance mismatch (Opening+Received-Sold+Adj != Closing)",
      stock_check[stock_check.expected_closing != stock_check.Closing_Stock])
check("Stock_Daily negative Opening/Closing", stock[(stock.Opening_Stock < 0) | (stock.Closing_Stock < 0)])
check("Stock_Daily duplicate (Product_ID, Date)", stock[stock.duplicated(["Product_ID", "Date"])])

check("Orders received before ordered", orders[orders.Received_Date.notna() & (orders.Received_Date < orders.Order_Date)])
check("Orders with non-positive Quantity_Ordered", orders[orders.Quantity_Ordered <= 0])
check("Orders.Product_ID not in Product_Master", orders[~orders.Product_ID.isin(products.Product_ID)])
check("Orders.Supplier_ID not in Suppliers", orders[~orders.Supplier_ID.isin(suppliers.Supplier_ID)])
check("Orders referencing inactive supplier", orders.merge(suppliers, on="Supplier_ID")[lambda x: x.Active == 0])
check("Orders with invalid Reason", orders[~orders.Reason.isin(
    ["Routine Replenishment", "Customer Request", "Stock Shortage", "Other"])])

check("Requests missing both Product_ID and Category", requests[requests.Requested_Product_ID.isna() & requests.Requested_Category.isna()])
check("Requests with invalid Outcome", requests[~requests.Outcome.isin(
    ["Sold", "Alternative Accepted", "Ordered and Fulfilled", "Unavailable", "Customer Declined", "Other"])])
check("Requests.Order_ID not in Orders", requests[requests.Order_ID.notna() & ~requests.Order_ID.isin(orders.Order_ID)])
check("Requests Fulfilled_Date before Request_Date", requests[requests.Fulfilled_Date.notna() & (requests.Fulfilled_Date < requests.Request_Date)])

check("Product_Master duplicate Product_ID", products[products.duplicated("Product_ID")])
check("Product_Master negative prices", products[(products.Unit_Cost < 0) | (products.Selling_Price < 0)])
check("Active products missing Supplier_ID", products[(products.Active == 1) & products.Supplier_ID.isna()])

print(f"{'Check':60s} Result")
print("-" * 75)
for name, status in issues:
    print(f"{name:60s} {status}")

print("\nCategory list (visual check for inconsistent naming):")
print(sorted(products.Category.unique()))

print("\nRoutinely_Stocked distribution:")
print(products.Routinely_Stocked.value_counts())

print("\nAntimalarial products (should be 1 stocked, 2 not):")
print(products[products.Category == "Antimalarial"][["Product_Name", "Routinely_Stocked"]])

print("\nSample: top 5 non-stocked products by request count (Q4/Q5 candidates):")
req_counts = requests[requests.Requested_Product_ID.notna()].groupby("Requested_Product_ID").size().sort_values(ascending=False)
top_nonstocked = req_counts[req_counts.index.isin(products[products.Routinely_Stocked == 0].Product_ID)].head(5)
print(products.set_index("Product_ID").loc[top_nonstocked.index, "Product_Name"].to_frame().assign(Request_Count=top_nonstocked.values))

print("\nUnavailable request rate overall:", round((requests.Available == 0).mean(), 3))
print("Orders unresolved (Received_Date null):", orders.Received_Date.isna().sum(), "of", len(orders))

# ... run your QC checks here ...

# after checks pass, export the validated copies
suppliers.to_csv(f"{CLEAN_DIR}/Suppliers.csv", index=False)
products.to_csv(f"{CLEAN_DIR}/Product_Master.csv", index=False)
sales.to_csv(f"{CLEAN_DIR}/Sales.csv", index=False)
stock.to_csv(f"{CLEAN_DIR}/Stock_Daily.csv", index=False)
orders.to_csv(f"{CLEAN_DIR}/Orders.csv", index=False)
requests.to_csv(f"{CLEAN_DIR}/Customer_Requests.csv", index=False)
print("Clean data saved to:", os.path.abspath(CLEAN_DIR))

Check                                                        Result
---------------------------------------------------------------------------
Duplicate Sale_ID                                            PASS
Sales.Product_ID not in Product_Master                       PASS
Sales with non-positive Quantity                             PASS
Sales.Sale_Date outside data window                          PASS
Sales for inactive products                                  PASS
Stock_Daily balance mismatch (Opening+Received-Sold+Adj != Closing) FLAG (1 rows)
Stock_Daily negative Opening/Closing                         PASS
Stock_Daily duplicate (Product_ID, Date)                     PASS
Orders received before ordered                               PASS
Orders with non-positive Quantity_Ordered                    PASS
Orders.Product_ID not in Product_Master                      PASS
Orders.Supplier_ID not in Suppliers                          PASS
Orders referencing inactive supplier            